# Step 4A: Fine-Tuning Bi-Encoder & Reranker

This notebook implements the fine-tuning pipeline for MediQuery:

1. **Phase 1** — Prepare training data (24K query–golden-chunk pairs + hard negatives)
2. **Phase 2** — Fine-tune the bi-encoder (`BAAI/bge-base-en-v1.5`)
3. **Phase 3** — Rebuild the FAISS index with fine-tuned embeddings
4. **Phase 4** — Fine-tune the cross-encoder reranker (`BAAI/bge-reranker-v2-m3`)
5. **Phase 5** — Retrieval evaluation (pre-trained vs. fine-tuned)

### Prerequisites

This notebook assumes that the **Step 2–3 notebook** has already been run, producing:

| Artifact | Path |
|---|---|
| Chunk corpus | `{DATA_DIR}/all_chunks.json` |
| Pre-trained FAISS index | `{DATA_DIR}/faiss_index/medicare.index` |
| Chunk metadata | `{DATA_DIR}/faiss_index/chunk_metadata.json` |
| Document IDs | `{DATA_DIR}/faiss_index/docids.txt` |
| Embeddings | `{DATA_DIR}/faiss_index/embeddings.npy` |

### Outputs

After running, the following are saved for **Step 4B** (End-to-End RAG Evaluation):

| Artifact | Path |
|---|---|
| Fine-tuned bi-encoder | `{DATA_DIR}/bge-base-mediquery-finetuned/` |
| Fine-tuned FAISS index | `{DATA_DIR}/faiss_index/medicare_finetuned.index` |
| Fine-tuned embeddings | `{DATA_DIR}/faiss_index/embeddings_finetuned.npy` |
| Fine-tuned reranker | `{DATA_DIR}/bge-reranker-mediquery-finetuned/` |

## Install Dependencies

In [2]:
!pip install -q sentence-transformers faiss-cpu pandas scikit-learn datasets

## Mount Google Drive & Path Configuration

All paths match the conventions from the Step 2–3 notebook.

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
import os

# ─── PATH CONFIGURATION (matches Step 2-3 notebook) ──────────────────────────
DATA_DIR    = '/content/drive/MyDrive/Embeddings2'
CHUNKS_FILE = f'{DATA_DIR}/all_chunks.json'
OUTPUT_DIR  = f'{DATA_DIR}/faiss_index'
INDEX_FILE  = f'{OUTPUT_DIR}/medicare.index'
DOCID_FILE  = f'{OUTPUT_DIR}/docids.txt'
EMBED_FILE  = f'{OUTPUT_DIR}/embeddings.npy'
META_FILE   = f'{OUTPUT_DIR}/chunk_metadata.json'

# ─── TRAINING & EVAL DATA ────────────────────────────────────────────────────
DATASETS_DIR     = f'{DATA_DIR}'
TRAIN_CSV        = f'{DATASETS_DIR}/query_golden_chunk_pairs_full_with_manuals.csv'

# ─── FINE-TUNED MODEL OUTPUT PATHS ───────────────────────────────────────────
FT_BIENCODER_DIR = f'{DATA_DIR}/bge-base-mediquery-finetuned'
FT_RERANKER_DIR  = f'{DATA_DIR}/bge-reranker-mediquery-finetuned'
FT_INDEX_FILE    = f'{OUTPUT_DIR}/medicare_finetuned.index'
FT_EMBED_FILE    = f'{OUTPUT_DIR}/embeddings_finetuned.npy'

os.makedirs(DATASETS_DIR, exist_ok=True)

print(f"Data directory:      {DATA_DIR}")
print(f"FAISS index dir:     {OUTPUT_DIR}")
print(f"Training CSV:        {TRAIN_CSV}")
print(f"FT bi-encoder dir:   {FT_BIENCODER_DIR}")
print(f"FT reranker dir:     {FT_RERANKER_DIR}")

Data directory:      /content/drive/MyDrive/Embeddings2
FAISS index dir:     /content/drive/MyDrive/Embeddings2/faiss_index
Training CSV:        /content/drive/MyDrive/Embeddings2/query_golden_chunk_pairs_full_with_manuals.csv
FT bi-encoder dir:   /content/drive/MyDrive/Embeddings2/bge-base-mediquery-finetuned
FT reranker dir:     /content/drive/MyDrive/Embeddings2/bge-reranker-mediquery-finetuned


## Upload Datasets

Upload the two CSV files to `{DATA_DIR}/datasets/` on Google Drive before running the next cell. Alternatively, upload them here with `files.upload()`.

In [5]:
# Option A: If files are already on Google Drive, just verify they exist
for path in [TRAIN_CSV]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  Found: {os.path.basename(path)} ({size_mb:.1f} MB)")
    else:
        print(f"  MISSING: {path}")
        print(f"  → Upload to Google Drive at: {DATASETS_DIR}/")

# Option B: Upload from local machine (uncomment if needed)
# from google.colab import files
# uploaded = files.upload()
# for name, data in uploaded.items():
#     dest = f'{DATASETS_DIR}/{name}'
#     with open(dest, 'wb') as f:
#         f.write(data)
#     print(f"Saved to {dest}")

  Found: query_golden_chunk_pairs_full_with_manuals.csv (60.1 MB)


## Load Pre-Trained Artifacts from Step 2–3

In [6]:
import json
import numpy as np
import faiss
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
from collections import Counter

# ─── Load chunk corpus ────────────────────────────────────────────────────────
with open(CHUNKS_FILE, 'r', encoding='utf-8') as f:
    all_chunks = json.load(f)
print(f"Chunks loaded: {len(all_chunks):,}")

# ─── Load chunk metadata ─────────────────────────────────────────────────────
with open(META_FILE, 'r', encoding='utf-8') as f:
    chunk_metadata = json.load(f)
print(f"Metadata loaded: {len(chunk_metadata):,}")

# ─── Load pre-trained FAISS index ─────────────────────────────────────────────
pretrained_index = faiss.read_index(INDEX_FILE)
print(f"Pre-trained FAISS index: {pretrained_index.ntotal:,} vectors, dim={pretrained_index.d}")

# ─── Device setup ─────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nDevice: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Chunks loaded: 8,563
Metadata loaded: 8,563
Pre-trained FAISS index: 8,563 vectors, dim=768

Device: cuda
GPU: NVIDIA A100-SXM4-40GB
GPU memory: 39.5 GB


## Helper Functions

Retrieval and reranking utilities used throughout the notebook. These mirror the functions from the Step 2–3 notebook but accept model/index as parameters so we can swap between pre-trained and fine-tuned versions.

In [ ]:
def retrieve_chunks(query, embed_model, index, all_chunks, chunk_metadata, top_k=100):
    """Embed a query and retrieve top-k chunks from a FAISS index."""
    query = "Represent this sentence for searching relevant passages: " + query
    query_vec = embed_model.encode(
        query, normalize_embeddings=True, convert_to_numpy=True
    ).astype('float32').reshape(1, -1)

    scores, indices = index.search(query_vec, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        if idx == -1:
            continue
        meta = chunk_metadata[idx]
        results.append({
            'faiss_score': float(scores[0][rank]),
            'text':        all_chunks[idx]['text'],
            'title':       meta['title'],
            'type':        meta['type'],
            'states':      meta.get('states', ['ALL']),
            'source_id':   meta['source_id'],
            'chunk_idx':   meta['chunk_idx'],
            'chunk_id':    f"{meta['source_id']}_{meta['chunk_idx']}"
        })
    return results


def filter_by_state(results, state=None):
    """Filter retrieved chunks to those covering a specific state."""
    if state is None:
        return results
    return [r for r in results if 'ALL' in r['states'] or state in r['states']]


def filter_claims_processing(results, query):
    coverage_keywords = ['cover', 'covered', 'coverage', 'eligible',
                         'eligibility', 'qualify', 'benefit']
    is_coverage_query = any(kw in query.lower() for kw in coverage_keywords)
    if is_coverage_query:
        return [r for r in results if r['type'] != 'CLAIMS_PROCESSING']
    return results  # keep everything for billing/documentation queries


def rerank_results(query, results, reranker_model, top_n=5, deduplicate=True):
    """Rerank retrieved chunks using a cross-encoder.

    Args:
        deduplicate: If True, keep only one chunk per source_id (for RAG generation).
                     If False, keep all chunks (for retrieval evaluation).
    """
    if not results:
        return []

    pairs = [(query, r['text']) for r in results]
    scores = reranker_model.predict(pairs)

    for i in range(len(results)):
        results[i]['rerank_score'] = float(scores[i])

    results.sort(key=lambda x: x['rerank_score'], reverse=True)

    if not deduplicate:
        return results[:top_n]

    seen = set()
    final = []
    for r in results:
        if r['source_id'] not in seen:
            final.append(r)
            seen.add(r['source_id'])
        if len(final) == top_n:
            break
    return final


def retrieve_and_rerank(query, embed_model, faiss_index, reranker_model,
                        all_chunks, chunk_metadata, state=None,
                        top_k_retrieve=100, top_n_rerank=5):
    """Full retrieval pipeline: embed -> FAISS -> state filter -> filter claims -> rerank."""
    retrieved = retrieve_chunks(query, embed_model, faiss_index,
                                all_chunks, chunk_metadata, top_k=top_k_retrieve)
    filtered = filter_by_state(retrieved, state)
    filtered = filter_claims_processing(filtered, query)
    reranked = rerank_results(query, filtered, reranker_model, top_n=top_n_rerank)
    return reranked


print('Helper functions defined.')

Helper functions defined.


---
# Phase 1: Prepare Training Data

Load the 24K query–golden-chunk pairs, construct hard negatives using the pre-trained FAISS index, and split into train/validation sets.

### Step 1.1 — Load and inspect the 24K pairs

In [8]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV)
print(f"Total training pairs: {len(train_df):,}")
print(f"\nColumns: {list(train_df.columns)}")
print(f"\n--- Query type distribution ---")
print(train_df['query_type'].value_counts())
print(f"\n--- Document type distribution ---")
print(train_df['type'].value_counts())
print(f"\n--- Sample row ---")
train_df.head(1)

Total training pairs: 56,288

Columns: ['query_id', 'query', 'query_type', 'gold_chunk_id', 'source_id', 'chunk_idx', 'title', 'type', 'coverage_status', 'states', 'filename', 'gold_chunk_excerpt', 'reference_answer']

--- Query type distribution ---
query_type
policy                 16424
summary                 8282
ncd_reference           8282
use_case                6316
criteria                6051
service_condition       5161
yes_no                  4135
indications              976
mixed_policy             489
non_coverage_reason      116
retired_status            56
Name: count, dtype: int64

--- Document type distribution ---
type
CLAIMS_PROCESSING    25411
LCD                  19365
BENEFIT_POLICY        6845
NCD                   4667
Name: count, dtype: int64

--- Sample row ---


,query_id,query,query_type,gold_chunk_id,source_id,chunk_idx,title,type,coverage_status,states,filename,gold_chunk_excerpt,reference_answer
0,100.3_0_q1,What is the Medicare coverage policy for 24-Ho...,policy,100.3_0,100.3,0,24-Hour Ambulatory Esophageal pH Monitoring,NCD,covered,ALL,24-Hour_Ambulatory_Esophageal_pH_Monitoring.txt,Twenty-four hour ambulatory pH monitoring is c...,Twenty-four hour ambulatory pH monitoring is c...


### Step 1.2 — Build (query, positive) pairs using actual indexed chunk text

**Critical:** The positive passage must be the **actual chunk text from `all_chunks.json`** (the same text that is embedded in the FAISS index), NOT the `gold_chunk_excerpt` from the CSV. The indexed chunks include metadata headers (e.g., `[TYPE: NCD | NCD_ID: 30.3.3 | ...]`) and are ~400 words long. Training on short excerpts would create a domain mismatch — the model would learn to embed short text well but fail on the long metadata-tagged chunks in the actual index.

In [9]:
# Build a lookup: chunk_id -> index in all_chunks for fast negative mining
chunk_id_to_idx = {}
for i, meta in enumerate(chunk_metadata):
    cid = f"{meta['source_id']}_{meta['chunk_idx']}"
    chunk_id_to_idx[cid] = i

print(f"Chunk ID lookup built: {len(chunk_id_to_idx):,} entries")

# Build training pairs using ACTUAL INDEXED CHUNK TEXT (not the short CSV excerpt)
train_pairs = []
skipped = 0
skipped_no_chunk = 0

for _, row in train_df.iterrows():
    gold_chunk_id = str(row['gold_chunk_id'])

    # Look up the actual chunk text from all_chunks.json
    if gold_chunk_id in chunk_id_to_idx:
        chunk_idx = chunk_id_to_idx[gold_chunk_id]
        positive_text = all_chunks[chunk_idx]['text']  # full metadata-tagged chunk
    else:
        # Chunk not found in index — skip this pair
        skipped_no_chunk += 1
        continue

    if len(positive_text.strip()) < 50:
        skipped += 1
        continue

    train_pairs.append({
        'query':          row['query'],
        'positive':       positive_text,
        'gold_chunk_id':  gold_chunk_id,
        'source_id':      str(row['source_id']),
        'query_type':     row['query_type'],
        'type':           row['type'],
        'coverage_status': row.get('coverage_status', ''),
    })

print(f"Valid training pairs: {len(train_pairs):,}")
print(f"Skipped (chunk not in index): {skipped_no_chunk}")
print(f"Skipped (short text): {skipped}")
print(f"\nSample positive text (first 200 chars):")
print(f"  {train_pairs[0]['positive'][:200]}...")

Chunk ID lookup built: 8,563 entries
Valid training pairs: 56,288
Skipped (chunk not in index): 0
Skipped (short text): 0

Sample positive text (first 200 chars):
  [TYPE: NCD | NCD_ID: 100.3 | TITLE: 24-Hour Ambulatory Esophageal pH Monitoring]

TITLE: 24-Hour Ambulatory Esophageal pH Monitoring SECTION: 100.3 EFFECTIVE DATE: 1985-06-11 00:00:00 COVERAGE POLICY:...


### Step 1.3 — Mine hard negatives from the pre-trained FAISS index

For each query, retrieve the top-300 chunks using the **pre-trained** FAISS index.
The highest-ranked chunk whose `source_id` differs from the gold `source_id` becomes
the hard negative. Following the [FlagEmbedding guide](https://github.com/FlagOpen/FlagEmbedding/tree/master/examples/finetune/embedder),
queries are prefixed with the BGE retrieval instruction during encoding to match
inference-time behavior.

In [10]:
from tqdm import tqdm

# Load the pre-trained embedding model for negative mining
pretrained_embed_model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)

print(f"Mining hard negatives for {len(train_pairs):,} queries...")
print("This encodes each query and searches the pre-trained FAISS index.\n")

# BGE query instruction — applied during training AND inference per FlagEmbedding guide
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "

# Batch-encode all queries WITH instruction prefix (matches inference-time behavior)
all_queries = [QUERY_INSTRUCTION + p['query'] for p in train_pairs]
query_embeddings = pretrained_embed_model.encode(
    all_queries,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

print(f"Query embeddings shape: {query_embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Mining hard negatives for 56,288 queries...
This encodes each query and searches the pre-trained FAISS index.



Batches:   0%|          | 0/220 [00:00<?, ?it/s]

Query embeddings shape: (56288, 768)


In [11]:
# Batch FAISS search for all queries at once
TOP_K_MINE = 300
scores_all, indices_all = pretrained_index.search(query_embeddings, TOP_K_MINE)

# Assign hard negatives
no_negative_count = 0
for i, pair in enumerate(tqdm(train_pairs, desc="Assigning hard negatives")):
    gold_source = pair['source_id']
    negative_text = None

    for idx in indices_all[i]:
        if idx == -1:
            continue
        meta = chunk_metadata[idx]
        if str(meta['source_id']) != gold_source:
            negative_text = all_chunks[idx]['text']
            break

    if negative_text is None:
        # Fallback: use the last retrieved chunk
        fallback_idx = indices_all[i][-1]
        if fallback_idx != -1:
            negative_text = all_chunks[fallback_idx]['text']
        else:
            negative_text = ""  # will be filtered out
        no_negative_count += 1

    pair['negative'] = negative_text

# Filter out any pairs with empty negatives
train_pairs = [p for p in train_pairs if len(p.get('negative', '')) > 20]

print(f"\nFinal training pairs with negatives: {len(train_pairs):,}")
print(f"Pairs where no distinct-source negative was found: {no_negative_count}")

Assigning hard negatives: 100%|██████████| 56288/56288 [00:00<00:00, 288714.40it/s]



Final training pairs with negatives: 56,288
Pairs where no distinct-source negative was found: 0


### Step 1.4 — Train / validation split

In [12]:
from sklearn.model_selection import train_test_split

# First, hold out 10% as a TEST set for final evaluation
query_types_all = [p['query_type'] for p in train_pairs]

trainval_set, test_set = train_test_split(
    train_pairs,
    test_size=0.10,
    random_state=42,
    stratify=query_types_all
)

print(f"Train+Val set: {len(trainval_set):,}")
print(f"Test set:      {len(test_set):,}")

# Then split train+val into train (95%) and val (5%) for training evaluation
query_types_tv = [p['query_type'] for p in trainval_set]

train_set, val_set = train_test_split(
    trainval_set,
    test_size=0.05,
    random_state=42,
    stratify=query_types_tv
)

print(f"Train set:     {len(train_set):,}")
print(f"Val set:       {len(val_set):,}")

Train+Val set: 50,659
Test set:      5,629
Train set:     48,126
Val set:       2,533


### Step 1.5 — Add manual chapter pairs to training data

The 24K training pairs come exclusively from NCDs and LCDs. The fine-tuned model becomes blind to **Claims Processing Manual** and **Benefit Policy Manual** chunks because it never sees them during training. We pull manual chapter queries from the 500-item eval CSV and add them as extra training pairs so the model learns what those chunks look like too.

In [13]:
# # Load the eval CSV to get manual chapter questions
# eval_df_early = pd.read_csv(EVAL_CSV)

# manual_extra_pairs = []

# for _, row in eval_df_early.iterrows():
#     if row['doc_type'] not in ['Medicare Claims Processing Manual',
#                                 'Medicare Benefit Policy Manual']:
#         continue

#     chunk_id = str(row['chunk_id'])
#     if chunk_id not in chunk_id_to_idx:
#         continue

#     idx = chunk_id_to_idx[chunk_id]
#     positive_text = all_chunks[idx]['text']

#     # Mine a hard negative using the pre-trained FAISS index
#     query_vec = pretrained_embed_model.encode(
#         row['question'], normalize_embeddings=True, convert_to_numpy=True
#     ).astype('float32').reshape(1, -1)

#     _, neg_indices = pretrained_index.search(query_vec, 20)
#     negative_text = None
#     gold_source = chunk_id.rsplit('_', 1)[0]

#     for neg_idx in neg_indices[0]:
#         if neg_idx == -1:
#             continue
#         if str(chunk_metadata[neg_idx]['source_id']) != gold_source:
#             negative_text = all_chunks[neg_idx]['text']
#             break

#     if negative_text is None:
#         continue

#     manual_extra_pairs.append({
#         'query':         row['question'],
#         'positive':      positive_text,
#         'negative':      negative_text,
#         'source_id':     gold_source,
#         'query_type':    row['question_type'],
#         'gold_chunk_id': chunk_id,
#     })

# print(f"Extra manual chapter pairs added: {len(manual_extra_pairs)}")

# # Combine with original train_set
# combined_train_set = train_set + manual_extra_pairs
combined_train_set = train_set
# print(f"Combined training set: {len(combined_train_set):,}")

### Step 1.6 — Create `Dataset` objects for bi-encoder training

The modern `sentence-transformers` Trainer API uses HuggingFace `datasets.Dataset` objects
instead of the legacy `InputExample` + `DataLoader` pattern. Each row has three columns:
`anchor` (query with instruction prefix), `positive` (gold chunk), and `negative` (hard negative).

Per the [FlagEmbedding fine-tuning guide](https://github.com/FlagOpen/FlagEmbedding/tree/master/examples/finetune/embedder),
the BGE retrieval instruction is prepended to queries during training
so the model learns representations consistent with inference-time behavior.

In [14]:
from datasets import Dataset

# BGE query instruction — prepended to queries during training per FlagEmbedding guide
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "

# Build HuggingFace Dataset with (anchor, positive, negative) columns
# Anchor = instruction + query (matches inference-time encoding in retrieve_chunks)
train_data = {
    'anchor':   [QUERY_INSTRUCTION + p['query'] for p in combined_train_set],
    'positive': [p['positive'] for p in combined_train_set],
    'negative': [p['negative'] for p in combined_train_set],
}
biencoder_train_dataset = Dataset.from_dict(train_data)

print(f"Bi-encoder training dataset: {len(biencoder_train_dataset):,} examples")
print(f"  Columns: {biencoder_train_dataset.column_names}")
print(f"  Sample anchor:   {biencoder_train_dataset[0]['anchor'][:150]}...")
print(f"  Sample positive: {biencoder_train_dataset[0]['positive'][:120]}...")
print(f"  Sample negative: {biencoder_train_dataset[0]['negative'][:120]}...")

Bi-encoder training examples: 48,126
  Sample query:    Summarize CMS guidance for Bladder/Urothelial Tumor Markers....
  Sample positive: [TYPE: LCD | LCD_ID: 36975 | TITLE: Bladder/Urothelial Tumor Markers | STATES: AL, AK, AZ, AR, CA, CO, CT, DE, DC, FL, G...
  Sample negative: [TYPE: LCD | LCD_ID: 36678 | TITLE: Lab: Bladder/Urothelial Tumor Markers | STATES: AL, AK, AZ, AR, CA, CO, CT, DE, DC, ...


---
# Phase 2: Fine-Tune the Bi-Encoder

Fine-tune `BAAI/bge-base-en-v1.5` using the modern **`SentenceTransformerTrainer`** API
with **`MultipleNegativesRankingLoss`** (MNRL). Following the
[FlagEmbedding fine-tuning guide](https://github.com/FlagOpen/FlagEmbedding/tree/master/examples/finetune/embedder):

- **Temperature 0.02** (`scale=50`): Matches BGE’s original contrastive training temperature
- **Query instruction**: Prepended to queries during training to match inference-time behavior
- **Trainer API**: `SentenceTransformerTrainer` (replaces legacy `model.fit()`)
- **`NO_DUPLICATES` sampler**: Ensures unique anchors per batch for maximum negative diversity
- **Warmup ratio 0.1**: 10% warmup as recommended by FlagEmbedding
- **`dataloader_drop_last=True`**: Drops incomplete final batch per FlagEmbedding defaults

### Step 2.1 — Load base model and configure loss

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import MultipleNegativesRankingLoss

# Load fresh base model for fine-tuning
biencoder_model = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)
print(f"Base model loaded: BAAI/bge-base-en-v1.5")
print(f"Embedding dim: {biencoder_model.get_sentence_embedding_dimension()}")

# Configure contrastive loss with temperature=0.02 (scale=1/0.02=50)
# This matches BGE's original training temperature per FlagEmbedding guide
train_loss = MultipleNegativesRankingLoss(biencoder_model, scale=50.0)
print(f"Loss: MultipleNegativesRankingLoss (scale=50, temperature=0.02)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Base model loaded: BAAI/bge-base-en-v1.5
Embedding dim: 768
Loss: MultipleNegativesRankingLoss
DataLoader: 1504 batches of 32


### Step 2.2 — Set up IR evaluator against the full corpus

The evaluator must search the full chunk corpus (all 8,563 chunks), not just the validation positives. Using only val positives as the corpus makes evaluation trivially easy — the model only has to pick the right answer from ~1,200 candidates instead of 8,563.

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

# Build the full corpus from all_chunks — every chunk the FAISS index contains.
full_ir_corpus = {str(i): chunk['text'] for i, chunk in enumerate(all_chunks)}

# Queries with instruction prefix (matches training and inference-time behavior)
ir_queries       = {str(i): QUERY_INSTRUCTION + p['query'] for i, p in enumerate(val_set)}
ir_relevant_docs = {str(i): {str(chunk_id_to_idx[p['gold_chunk_id']])}
                    for i, p in enumerate(val_set)}

biencoder_evaluator = InformationRetrievalEvaluator(
    ir_queries,
    full_ir_corpus,
    ir_relevant_docs,
    name='mediquery-biencoder-val',
    show_progress_bar=True
)

print(f"IR Evaluator: {len(ir_queries)} queries against {len(full_ir_corpus)} corpus docs")

IR Evaluator: 2533 queries against 8563 corpus docs


### Step 2.3 — Train the bi-encoder with SentenceTransformerTrainer

Uses the modern `SentenceTransformerTrainer` API with parameters aligned to the
[FlagEmbedding fine-tuning guide](https://github.com/FlagOpen/FlagEmbedding/tree/master/examples/finetune/embedder).

| Parameter | Value | Rationale |
|---|---|---|
| Epochs | 2 | Standard for BGE fine-tuning per FlagEmbedding examples |
| Learning rate | 1e-5 | Matches FlagEmbedding default for encoder-only models |
| Batch size | 32 | Larger = more in-batch negatives for MNRL |
| Warmup ratio | 0.1 | 10% warmup per FlagEmbedding default |
| Temperature | 0.02 (scale=50) | Matches BGE’s original contrastive training |
| Batch sampler | NO_DUPLICATES | Prevents duplicate anchors per batch |
| dataloader_drop_last | True | Drops incomplete final batch per FlagEmbedding |
| Mixed precision | fp16 | Matches FlagEmbedding `fp16=True` default |

In [ ]:
from sentence_transformers import SentenceTransformerTrainer
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.training_args import BatchSamplers

EPOCHS = 2
LR = 1e-5
BATCH_SIZE = 32  # Adjust: T4 -> 32, A100 -> 64-128
EVAL_STEPS = 500

args = SentenceTransformerTrainingArguments(
    output_dir=FT_BIENCODER_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,                          # 10% warmup per FlagEmbedding
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    dataloader_drop_last=True,                 # per FlagEmbedding defaults
    load_best_model_at_end=True,
    logging_steps=100,
)

trainer = SentenceTransformerTrainer(
    model=biencoder_model,
    args=args,
    train_dataset=biencoder_train_dataset,
    loss=train_loss,
    evaluator=biencoder_evaluator,
)

print(f"Training plan:")
print(f"  Epochs:         {EPOCHS}")
print(f"  Batch size:     {BATCH_SIZE}")
print(f"  LR:             {LR}")
print(f"  Warmup ratio:   0.1 (10%)")
print(f"  Temperature:    0.02 (scale=50)")
print(f"  Eval every:     {EVAL_STEPS} steps")
print(f"  Batch sampler:  NO_DUPLICATES")
print(f"  Drop last batch: True")
print(f"  Output:         {FT_BIENCODER_DIR}")
print(f"\nStarting training...\n")

trainer.train()

# Save the final model
biencoder_model.save_pretrained(FT_BIENCODER_DIR)

print(f"\nBi-encoder fine-tuning complete!")
print(f"Best model saved to: {FT_BIENCODER_DIR}")

Training plan:
  Epochs:       2
  Steps/epoch:  1504
  Total steps:  3008
  Warmup:       200
  LR:           1e-05
  Eval every:   500 steps
  Output:       /content/drive/MyDrive/Embeddings2/bge-base-mediquery-finetuned

Starting training...



Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Mediquery-biencoder-val Cosine Accuracy@1,Mediquery-biencoder-val Cosine Accuracy@3,Mediquery-biencoder-val Cosine Accuracy@5,Mediquery-biencoder-val Cosine Accuracy@10,Mediquery-biencoder-val Cosine Precision@1,Mediquery-biencoder-val Cosine Precision@3,Mediquery-biencoder-val Cosine Precision@5,Mediquery-biencoder-val Cosine Precision@10,Mediquery-biencoder-val Cosine Recall@1,Mediquery-biencoder-val Cosine Recall@3,Mediquery-biencoder-val Cosine Recall@5,Mediquery-biencoder-val Cosine Recall@10,Mediquery-biencoder-val Cosine Ndcg@10,Mediquery-biencoder-val Cosine Mrr@10,Mediquery-biencoder-val Cosine Map@100
500,1.066890,No log,0.170154,0.333202,0.421240,0.541255,0.170154,0.111067,0.084248,0.054126,0.170154,0.333202,0.421240,0.541255,0.339717,0.277142,0.289002
1000,0.625303,No log,0.210028,0.386498,0.478484,0.589420,0.210028,0.128833,0.095697,0.058942,0.210028,0.386498,0.478484,0.589420,0.386420,0.322994,0.334722
1500,0.492773,No log,0.225424,0.404264,0.500197,0.613502,0.225424,0.134755,0.100039,0.061350,0.225424,0.404264,0.500197,0.613502,0.404982,0.339872,0.351389
1504,0.492773,No log,0.227398,0.403869,0.501382,0.615476,0.227398,0.134623,0.100276,0.061548,0.227398,0.403869,0.501382,0.615476,0.406253,0.341016,0.352403
2000,0.403785,No log,0.236478,0.420450,0.510857,0.627319,0.236478,0.140150,0.102171,0.062732,0.236478,0.420450,0.510857,0.627319,0.418467,0.353139,0.364427
2500,0.374892,No log,0.239637,0.431109,0.523885,0.634820,0.239637,0.143703,0.104777,0.063482,0.239637,0.431109,0.523885,0.634820,0.424563,0.358702,0.369926
3000,0.366704,No log,0.240426,0.429530,0.526648,0.637189,0.240426,0.143177,0.105330,0.063719,0.240426,0.429530,0.526648,0.637189,0.425975,0.359811,0.371123
3008,0.366704,No log,0.240032,0.430320,0.526648,0.636794,0.240032,0.143440,0.105330,0.063679,0.240032,0.430320,0.526648,0.636794,0.425751,0.359611,0.370960


Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/268 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:12<00:00, 12.06s/it]



Bi-encoder fine-tuning complete!
Best model saved to: /content/drive/MyDrive/Embeddings2/bge-base-mediquery-finetuned


### Step 2.4 — Load and verify the fine-tuned bi-encoder

In [ ]:
finetuned_embed_model = SentenceTransformer(FT_BIENCODER_DIR, device=device)
print(f"Fine-tuned bi-encoder loaded from: {FT_BIENCODER_DIR}")
print(f"Embedding dim: {finetuned_embed_model.get_sentence_embedding_dimension()}")

# Quick verification: encode a test query
test_emb = finetuned_embed_model.encode("test", normalize_embeddings=True)
print(f"L2 norm check: {np.linalg.norm(test_emb):.6f} (should be 1.0)")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fine-tuned bi-encoder loaded from: /content/drive/MyDrive/Embeddings2/bge-base-mediquery-finetuned
Embedding dim: 768
L2 norm check: 1.000000 (should be 1.0)


---
# Phase 3: Rebuild FAISS Index with Fine-Tuned Embeddings

### Step 3.1 — Re-encode all 8,563 chunks

In [18]:
texts = [chunk['text'] for chunk in all_chunks]

print(f"Re-encoding {len(texts):,} chunks with fine-tuned bi-encoder...")

finetuned_embeddings = finetuned_embed_model.encode(
    texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"\nEmbeddings shape: {finetuned_embeddings.shape}")
print(f"Dtype: {finetuned_embeddings.dtype}")

norms = np.linalg.norm(finetuned_embeddings, axis=1)
print(f"L2 norm check — mean: {norms.mean():.6f}, min: {norms.min():.6f}, max: {norms.max():.6f}")

Re-encoding 8,563 chunks with fine-tuned bi-encoder...


Batches:   0%|          | 0/134 [00:00<?, ?it/s]


Embeddings shape: (8563, 768)
Dtype: float32
L2 norm check — mean: 1.000000, min: 1.000000, max: 1.000000


### Step 3.2 — Build and save the fine-tuned FAISS index

In [19]:
dimension = finetuned_embeddings.shape[1]
finetuned_index = faiss.IndexFlatIP(dimension)
finetuned_index.add(finetuned_embeddings.astype('float32'))

faiss.write_index(finetuned_index, FT_INDEX_FILE)
np.save(FT_EMBED_FILE, finetuned_embeddings)

print(f"Fine-tuned FAISS index saved: {FT_INDEX_FILE}")
print(f"  Vectors: {finetuned_index.ntotal:,}")
print(f"  Dimension: {finetuned_index.d}")
print(f"Fine-tuned embeddings saved: {FT_EMBED_FILE}")
print(f"  Size: {os.path.getsize(FT_INDEX_FILE) / 1024**2:.1f} MB")

Fine-tuned FAISS index saved: /content/drive/MyDrive/Embeddings2/faiss_index/medicare_finetuned.index
  Vectors: 8,563
  Dimension: 768
Fine-tuned embeddings saved: /content/drive/MyDrive/Embeddings2/faiss_index/embeddings_finetuned.npy
  Size: 25.1 MB


### Step 3.3 — Sanity check: compare retrieval on 3 test queries

In [20]:
test_queries = [
    "Does Medicare cover acupuncture for chronic lower back pain?",
    "Is home oxygen therapy covered for patients in Texas?",
    "What are the requirements for skilled nursing facility coverage?"
]

print("=" * 95)
print("RETRIEVAL COMPARISON: Pre-trained vs. Fine-tuned Bi-Encoder (top-3)")
print("=" * 95)

for query in test_queries:
    print(f"\nQuery: \"{query}\"")
    print("-" * 95)

    # Pre-trained
    pt_results = retrieve_chunks(query, pretrained_embed_model, pretrained_index,
                                  all_chunks, chunk_metadata, top_k=3)
    # Fine-tuned
    ft_results = retrieve_chunks(query, finetuned_embed_model, finetuned_index,
                                  all_chunks, chunk_metadata, top_k=3)

    print(f"  {'Pre-trained':^45s} | {'Fine-tuned':^45s}")
    print(f"  {'-'*45} | {'-'*45}")
    for rank in range(3):
        pt = pt_results[rank] if rank < len(pt_results) else {'title': '—', 'faiss_score': 0}
        ft = ft_results[rank] if rank < len(ft_results) else {'title': '—', 'faiss_score': 0}
        pt_str = f"#{rank+1} {pt['title'][:30]:30s} ({pt['faiss_score']:.4f})"
        ft_str = f"#{rank+1} {ft['title'][:30]:30s} ({ft['faiss_score']:.4f})"
        print(f"  {pt_str:45s} | {ft_str:45s}")

RETRIEVAL COMPARISON: Pre-trained vs. Fine-tuned Bi-Encoder (top-3)

Query: "Does Medicare cover acupuncture for chronic lower back pain?"
-----------------------------------------------------------------------------------------------
                   Pre-trained                  |                  Fine-tuned                  
  --------------------------------------------- | ---------------------------------------------
  #1 Acupuncture for Chronic Lower  (0.8095)    | #1 Acupuncture for Chronic Lower  (0.7071)   
  #2 Acupuncture                    (0.7742)    | #2 Acupuncture                    (0.4887)   
  #3 Acupuncture for Fibromyalgia   (0.7358)    | #3 Spinal Cord Stimulators for Ch (0.4866)   

Query: "Is home oxygen therapy covered for patients in Texas?"
-----------------------------------------------------------------------------------------------
                   Pre-trained                  |                  Fine-tuned                  
  ---------------------------

---
# Phase 4: Fine-Tune the Cross-Encoder Reranker

The reranker is the final precision gate — it decides which 5 chunks reach the LLM. Fine-tuning it on Medicare-specific relevance judgments teaches the model to better distinguish between truly relevant policy text and semantically similar but irrelevant passages.

### Step 4.1 — Re-mine hard negatives from the fine-tuned FAISS index

The bi-encoder hard negatives mined in Phase 1 came from the **pre-trained** index.
Now that we have a fine-tuned index, we re-mine harder negatives — passages the
fine-tuned bi-encoder ranks highly but that come from a different source than the
gold chunk.

Per the [FlagEmbedding guide](https://github.com/FlagOpen/FlagEmbedding/tree/master/examples/finetune/reranker),
queries are prefixed with the BGE retrieval instruction when encoding (consistent
with bi-encoder training). The resulting training data is formatted as a
`datasets.Dataset` with `(sentence1, sentence2, label)` columns for the
`CrossEncoderTrainer`.

In [ ]:
from datasets import Dataset

# Re-mine hard negatives using the fine-tuned bi-encoder + fine-tuned FAISS index
reranker_train_set = train_set

print("Re-mining hard negatives from the fine-tuned FAISS index...\n")

# BGE query instruction — must match bi-encoder training for consistent embeddings
QUERY_INSTRUCTION = "Represent this sentence for searching relevant passages: "

# Batch-encode all training queries WITH instruction prefix
train_queries = [QUERY_INSTRUCTION + p['query'] for p in reranker_train_set]
ft_query_embeddings = finetuned_embed_model.encode(
    train_queries,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

# Search fine-tuned FAISS index
NUM_HARD_NEGATIVES = 7  # train_group_size=8 per FlagEmbedding (1 pos + 7 neg)
TOP_K_MINE_FT = 100
scores_ft, indices_ft = finetuned_index.search(ft_query_embeddings, TOP_K_MINE_FT)

# Assign hard negatives per training query
neg_counts = []
for i, pair in enumerate(tqdm(reranker_train_set, desc="Mining FT hard negatives")):
    gold_source = pair['source_id']
    negatives = []
    seen_sources = {gold_source}

    for idx in indices_ft[i]:
        if idx == -1:
            continue
        meta = chunk_metadata[idx]
        src = str(meta['source_id'])
        if src in seen_sources:
            continue
        seen_sources.add(src)
        negatives.append(all_chunks[idx]['text'])
        if len(negatives) == NUM_HARD_NEGATIVES:
            break

    # Fallback: pad with non-source-deduped candidates
    if len(negatives) < NUM_HARD_NEGATIVES:
        for idx in indices_ft[i]:
            if idx == -1:
                continue
            meta = chunk_metadata[idx]
            if str(meta['source_id']) == gold_source:
                continue
            text = all_chunks[idx]['text']
            if text not in negatives:
                negatives.append(text)
            if len(negatives) == NUM_HARD_NEGATIVES:
                break

    pair['ft_negatives'] = negatives
    pair['ft_negative'] = negatives[0] if negatives else pair['negative']
    neg_counts.append(len(negatives))

print(f"\nRe-mined hard negatives for {len(reranker_train_set):,} training pairs")
print(f"Negatives per query: min={min(neg_counts)}, max={max(neg_counts)}, avg={sum(neg_counts)/len(neg_counts):.1f}")

# Also re-mine for validation set
val_queries = [QUERY_INSTRUCTION + p['query'] for p in val_set]
ft_val_embeddings = finetuned_embed_model.encode(
    val_queries,
    batch_size=256,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
).astype('float32')

scores_val_ft, indices_val_ft = finetuned_index.search(ft_val_embeddings, TOP_K_MINE_FT)

for i, pair in enumerate(tqdm(val_set, desc="Mining FT val hard negatives")):
    gold_source = pair['source_id']
    negatives = []
    seen_sources = {gold_source}

    for idx in indices_val_ft[i]:
        if idx == -1:
            continue
        meta = chunk_metadata[idx]
        src = str(meta['source_id'])
        if src in seen_sources:
            continue
        seen_sources.add(src)
        negatives.append(all_chunks[idx]['text'])
        if len(negatives) == NUM_HARD_NEGATIVES:
            break

    pair['ft_negatives'] = negatives
    pair['ft_negative'] = negatives[0] if negatives else pair['negative']

# Build reranker training Dataset: (sentence1, sentence2, label)
# 1 positive + up to N negatives per query (flat pairwise format)
reranker_rows = {'sentence1': [], 'sentence2': [], 'label': []}
skipped_train = 0

for pair in reranker_train_set:
    if not pair.get('ft_negatives'):
        skipped_train += 1
        continue
    # Positive pair
    reranker_rows['sentence1'].append(pair['query'])
    reranker_rows['sentence2'].append(pair['positive'])
    reranker_rows['label'].append(1.0)
    # Negative pairs
    for neg_text in pair['ft_negatives']:
        reranker_rows['sentence1'].append(pair['query'])
        reranker_rows['sentence2'].append(neg_text)
        reranker_rows['label'].append(0.0)

reranker_train_dataset = Dataset.from_dict(reranker_rows)

print(f"\nReranker training dataset: {len(reranker_train_dataset):,} examples")
print(f"(Skipped {skipped_train} training queries due to missing hard negatives)")

Re-mining hard negatives from the fine-tuned FAISS index...



Batches:   0%|          | 0/188 [00:00<?, ?it/s]

Mining FT hard negatives: 100%|██████████| 48126/48126 [00:01<00:00, 30372.73it/s]


Re-mined hard negatives for 48,126 training pairs
Negatives per query: min=0, max=5, avg=4.0


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Mining FT val hard negatives: 100%|██████████| 2533/2533 [00:00<00:00, 38053.42it/s]



Reranker training examples: 232,197
(Skipped 9016 training queries due to missing hard negatives)
Reranker validation examples: 12,508


### Step 4.2 — Load base cross-encoder and configure loss

Load `BAAI/bge-reranker-v2-m3` and configure `BinaryCrossEntropyLoss` for
pairwise relevance training (label 1.0 = relevant, 0.0 = irrelevant).
Following the [FlagEmbedding reranker guide](https://huggingface.co/BAAI/bge-reranker-v2-m3#fine-tune)
and the modern `CrossEncoderTrainer` API.

In [ ]:
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss

# Load base reranker
reranker_model = CrossEncoder('BAAI/bge-reranker-v2-m3', device=device, max_length=512)
print(f"Base reranker loaded: BAAI/bge-reranker-v2-m3")

# Configure loss: Binary cross-entropy for pairwise relevance labels
reranker_loss = BinaryCrossEntropyLoss(model=reranker_model)
print(f"Loss: BinaryCrossEntropyLoss")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Base reranker loaded: BAAI/bge-reranker-v2-m3
DataLoader: 7257 batches of 32


### Step 4.3 — Set up reranker evaluation callback

Uses `CrossEncoderRerankingEvaluator` to compute NDCG during training.
Each sample has 1 positive + N hard negatives for ranking evaluation.

In [ ]:
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator

# Build eval samples for CrossEncoderRerankingEvaluator (using fine-tuned negatives)
# Skip samples with no negatives — evaluator requires at least 2 docs per query.
reranker_eval_samples = []
skipped = 0
for pair in val_set:
    if not pair.get('ft_negatives'):
        skipped += 1
        continue
    reranker_eval_samples.append({
        'query':    pair['query'],
        'positive': [pair['positive']],
        'negative': pair['ft_negatives']
    })

reranker_evaluator = CrossEncoderRerankingEvaluator(
    reranker_eval_samples,
    name='mediquery-reranker-val'
)

print(f"Reranker evaluator: {len(reranker_eval_samples)} samples (skipped {skipped} with no negatives)")

Reranker evaluator: 2046 samples (skipped 487 with no negatives)


/tmp/ipykernel_3339/703104181.py:16: DeprecationWarning: The CERerankingEvaluator has been renamed to CrossEncoderCorrelationEvaluator. Please use CrossEncoderCorrelationEvaluator instead.
  reranker_evaluator = CERerankingEvaluator(


### Step 4.4 — Train the reranker with CrossEncoderTrainer

Uses the modern `CrossEncoderTrainer` API with parameters aligned to the
[FlagEmbedding encoder-only reranker example](https://github.com/FlagOpen/FlagEmbedding/tree/master/examples/finetune/reranker).

| Parameter | Value | Rationale |
|---|---|---|
| Epochs | 2 | Matches FlagEmbedding encoder-only reranker example |
| Learning rate | 6e-5 | FlagEmbedding default for encoder-only reranker |
| Batch size | 32 | Pairwise examples; effective group via 1 pos + 7 neg ratio |
| Warmup ratio | 0.1 | 10% warmup per FlagEmbedding |
| Weight decay | 0.01 | Regularization per FlagEmbedding |
| Gradient checkpointing | True | Memory efficiency per FlagEmbedding |
| dataloader_drop_last | True | Drops incomplete final batch per FlagEmbedding |
| Mixed precision | fp16 | Matches FlagEmbedding `fp16=True` default |

In [ ]:
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments

RERANKER_EPOCHS = 2           # FlagEmbedding encoder-only reranker uses 2
RERANKER_LR = 6e-5            # FlagEmbedding default for encoder-only reranker
RERANKER_BATCH_SIZE = 32
RERANKER_EVAL_STEPS = 2000

reranker_args = CrossEncoderTrainingArguments(
    output_dir=FT_RERANKER_DIR,
    num_train_epochs=RERANKER_EPOCHS,
    per_device_train_batch_size=RERANKER_BATCH_SIZE,
    learning_rate=RERANKER_LR,
    warmup_ratio=0.1,                          # 10% warmup per FlagEmbedding
    weight_decay=0.01,                         # per FlagEmbedding
    eval_strategy="steps",
    eval_steps=RERANKER_EVAL_STEPS,
    save_strategy="steps",
    save_steps=RERANKER_EVAL_STEPS,
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,               # per FlagEmbedding
    dataloader_drop_last=True,                 # per FlagEmbedding
    load_best_model_at_end=True,
    logging_steps=100,
)

reranker_trainer = CrossEncoderTrainer(
    model=reranker_model,
    args=reranker_args,
    train_dataset=reranker_train_dataset,
    loss=reranker_loss,
    evaluator=reranker_evaluator,
)

print(f"Reranker training plan:")
print(f"  Epochs:               {RERANKER_EPOCHS}")
print(f"  Batch size:           {RERANKER_BATCH_SIZE}")
print(f"  LR:                   {RERANKER_LR}")
print(f"  Warmup ratio:         0.1 (10%)")
print(f"  Weight decay:         0.01")
print(f"  Gradient checkpointing: True")
print(f"  Eval every:           {RERANKER_EVAL_STEPS} steps")
print(f"  Drop last batch:      True")
print(f"  Output:               {FT_RERANKER_DIR}")
print(f"\nStarting reranker training...\n")

reranker_trainer.train()

# Save the final model
reranker_model.save_pretrained(FT_RERANKER_DIR)

print(f"\nReranker fine-tuning complete!")
print(f"Best model saved to: {FT_RERANKER_DIR}")

Reranker training plan:
  Epochs:       1
  Steps/epoch:  7257
  Total steps:  7257
  Warmup:       100
  LR:           5e-06
  Output:       /content/drive/MyDrive/Embeddings2/bge-reranker-mediquery-finetuned

Starting reranker training...



Token indices sequence length is longer than the specified maximum sequence length for this model (641 > 512). Running this sequence through the model will result in indexing errors


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

### Step 4.5 — Load and verify the fine-tuned reranker

In [ ]:
finetuned_reranker = CrossEncoder(FT_RERANKER_DIR, device=device, max_length=512)
print(f"Fine-tuned reranker loaded from: {FT_RERANKER_DIR}")

# Sanity check: score a relevant vs irrelevant pair
# Use a real chunk from the corpus for the test
acupuncture_chunk = None
wound_chunk = None
for c in all_chunks:
    if '30.3.3' in c.get('text', '') and acupuncture_chunk is None:
        acupuncture_chunk = c['text'][:500]
    if 'Wound Care' in c.get('text', '') and wound_chunk is None:
        wound_chunk = c['text'][:500]
    if acupuncture_chunk and wound_chunk:
        break

test_query = "Does Medicare cover acupuncture for chronic lower back pain?"
scores = finetuned_reranker.predict([
    (test_query, acupuncture_chunk),
    (test_query, wound_chunk)
])

print(f"\nSanity check — query: \"{test_query}\"")
print(f"  Relevant (acupuncture NCD):  {scores[0]:.4f}")
print(f"  Irrelevant (wound care LCD): {scores[1]:.4f}")
print(f"  Relevant > Irrelevant: {scores[0] > scores[1]}")

---
# Phase 5: Retrieval Evaluation (Pre-trained vs. Fine-tuned)

Evaluate retrieval quality on the held-out **500 QA eval set** using Recall@k and MRR.

### Step 5.1 — Build test set from held-out training pairs

In [ ]:
# Build eval_df from the held-out test set (split from query_golden_chunk_pairs_full_with_manuals.csv)
eval_df = pd.DataFrame(test_set)

# Rename columns to match the evaluation function's expected schema
eval_df = eval_df.rename(columns={
    'query':         'question',
    'gold_chunk_id': 'chunk_id',
    'query_type':    'question_type',
    'type':          'doc_type',
})

print(f"Evaluation set loaded: {len(eval_df)} questions (held-out test split)")
print(f"Columns: {list(eval_df.columns)}")
print(f"--- Question type distribution ---")
print(eval_df['question_type'].value_counts())
print(f"--- Document type distribution ---")
print(eval_df['doc_type'].value_counts())
if 'coverage_status' in eval_df.columns:
    print(f"--- Coverage status distribution ---")
    print(eval_df['coverage_status'].value_counts())

### Step 5.2 — Evaluate retrieval for both models

In [ ]:
import math

def ndcg_at_k(gold_id, ranked_ids, k):
    """Compute NDCG@k for a single query with one relevant document (binary relevance).

    With a single relevant doc the ideal DCG is always 1.0 (relevant doc at rank 1),
    so NDCG@k = 1/log2(rank+1) if the gold doc appears in the top-k, else 0.
    """
    try:
        rank = ranked_ids[:k].index(gold_id)  # 0-based
        return 1.0 / math.log2(rank + 2)      # +2 because rank is 0-based
    except ValueError:
        return 0.0


def evaluate_retrieval(embed_model, faiss_index, reranker_model,
                       all_chunks, chunk_metadata, eval_df,
                       top_k_retrieve=100, top_n_rerank=5,
                       apply_claims_filter=False):
    """
    Evaluate retrieval performance on the eval set.

    Returns a dict with Recall@k, MRR, NDCG@k (before and after reranking),
    and per-row details for breakdown analysis.

    Note:
    For fair reranker evaluation, keep the same FAISS candidate pool for both
    FAISS and reranked metrics. Optional heuristics (claims filtering) can be
    enabled explicitly via apply_claims_filter.
    """
    results_per_row = []

    for _, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc="Evaluating"):
        question = row['question']
        gold_chunk_id = str(row['chunk_id'])

        # Retrieve top-k candidates from FAISS
        retrieved = retrieve_chunks(
            question, embed_model, faiss_index,
            all_chunks, chunk_metadata, top_k=top_k_retrieve
        )
        retrieved_ids = [r['chunk_id'] for r in retrieved]

        # Keep reranker candidates aligned with FAISS metrics by default.
        candidates_for_rerank = retrieved
        if apply_claims_filter:
            candidates_for_rerank = filter_claims_processing(retrieved, question)

        # Rerank to top-5 (no dedup for fair eval comparison)
        reranked = rerank_results(
            question,
            candidates_for_rerank,
            reranker_model,
            top_n=top_n_rerank,
            deduplicate=False
        )
        reranked_ids = [r['chunk_id'] for r in reranked]

        # --- Recall@k (before reranking) ---
        hit_at_5  = 1 if gold_chunk_id in retrieved_ids[:5]  else 0
        hit_at_10 = 1 if gold_chunk_id in retrieved_ids[:10] else 0
        hit_at_20 = 1 if gold_chunk_id in retrieved_ids[:20] else 0
        hit_at_50 = 1 if gold_chunk_id in retrieved_ids[:50] else 0
        hit_at_100 = 1 if gold_chunk_id in retrieved_ids[:100] else 0

        # --- MRR (before reranking) ---
        if gold_chunk_id in retrieved_ids:
            mrr = 1.0 / (retrieved_ids.index(gold_chunk_id) + 1)
        else:
            mrr = 0.0

        # --- NDCG@k (before reranking) ---
        ndcg_5  = ndcg_at_k(gold_chunk_id, retrieved_ids, 5)
        ndcg_10 = ndcg_at_k(gold_chunk_id, retrieved_ids, 10)
        ndcg_20 = ndcg_at_k(gold_chunk_id, retrieved_ids, 20)

        # --- Recall@5 after reranking ---
        hit_at_5_reranked = 1 if gold_chunk_id in reranked_ids else 0

        # --- MRR after reranking ---
        if gold_chunk_id in reranked_ids:
            mrr_reranked = 1.0 / (reranked_ids.index(gold_chunk_id) + 1)
        else:
            mrr_reranked = 0.0

        # --- NDCG@5 after reranking ---
        ndcg_5_reranked = ndcg_at_k(gold_chunk_id, reranked_ids, 5)

        results_per_row.append({
            'qa_id':              row.get('qa_id', ''),
            'question_type':      row.get('question_type', ''),
            'doc_type':           row.get('doc_type', ''),
            'coverage_status':    row.get('coverage_status', ''),
            'hit_at_5':           hit_at_5,
            'hit_at_10':          hit_at_10,
            'hit_at_20':          hit_at_20,
            'hit_at_50':          hit_at_50,
            'hit_at_100':         hit_at_100,
            'mrr':                mrr,
            'ndcg_5':             ndcg_5,
            'ndcg_10':            ndcg_10,
            'ndcg_20':            ndcg_20,
            'hit_at_5_reranked':  hit_at_5_reranked,
            'mrr_reranked':       mrr_reranked,
            'ndcg_5_reranked':    ndcg_5_reranked,
        })

    results_df = pd.DataFrame(results_per_row)

    summary = {
        'Recall@5 (FAISS)':    results_df['hit_at_5'].mean(),
        'Recall@10 (FAISS)':   results_df['hit_at_10'].mean(),
        'Recall@20 (FAISS)':   results_df['hit_at_20'].mean(),
        'Recall@50 (FAISS)':   results_df['hit_at_50'].mean(),
        'Recall@100 (FAISS)':  results_df['hit_at_100'].mean(),
        'MRR (FAISS)':         results_df['mrr'].mean(),
        'NDCG@5 (FAISS)':      results_df['ndcg_5'].mean(),
        'NDCG@10 (FAISS)':     results_df['ndcg_10'].mean(),
        'NDCG@20 (FAISS)':     results_df['ndcg_20'].mean(),
        'Recall@5 (reranked)': results_df['hit_at_5_reranked'].mean(),
        'MRR (reranked)':      results_df['mrr_reranked'].mean(),
        'NDCG@5 (reranked)':   results_df['ndcg_5_reranked'].mean(),
    }

    return summary, results_df


print("Evaluation function defined.")

In [ ]:
# Also load the pre-trained reranker for comparison
pretrained_reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=device, max_length=512)
print("Pre-trained reranker loaded for comparison.")

In [ ]:
eval_df = eval_df[:500]

In [33]:
# from sentence_transformers import CrossEncoder

# # Load pretrained reranker as placeholder (we only care about FAISS metrics)
# placeholder_reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=device, max_length=512)

# print("Evaluating FINE-TUNED bi-encoder ONLY (FAISS retrieval, no reranking)...\n")
# ft_biencoder_summary, ft_biencoder_details = evaluate_retrieval(
#     finetuned_embed_model, finetuned_index, placeholder_reranker,
#     all_chunks, chunk_metadata, eval_df,
#     top_k_retrieve=10, top_n_rerank=5
# )

# print("\n--- Fine-tuned Bi-Encoder Only (FAISS metrics) ---")
# for metric in ['Recall@5 (FAISS)', 'Recall@10 (FAISS)', 'Recall@20 (FAISS)',
#                'MRR (FAISS)', 'NDCG@5 (FAISS)', 'NDCG@10 (FAISS)', 'NDCG@20 (FAISS)']:
#     print(f"  {metric:25s}: {ft_biencoder_summary[metric]:.4f}")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Evaluating FINE-TUNED bi-encoder ONLY (FAISS retrieval, no reranking)...



Evaluating: 100%|██████████| 500/500 [01:52<00:00,  4.43it/s]


--- Fine-tuned Bi-Encoder Only (FAISS metrics) ---
  Recall@5 (FAISS)         : 0.5320
  Recall@10 (FAISS)        : 0.6560
  Recall@20 (FAISS)        : 0.6560
  MRR (FAISS)              : 0.3689
  NDCG@5 (FAISS)           : 0.3969
  NDCG@10 (FAISS)          : 0.4370
  NDCG@20 (FAISS)          : 0.4370


In [35]:
# from sentence_transformers import CrossEncoder

# # Load pretrained reranker as placeholder (we only care about FAISS metrics)
# placeholder_reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=device, max_length=512)

# print("Evaluating FINE-TUNED bi-encoder ONLY (FAISS retrieval, no reranking)...\n")
# ft_biencoder_summary, ft_biencoder_details = evaluate_retrieval(
#     pretrained_embed_model, pretrained_index, pretrained_reranker,
#     all_chunks, chunk_metadata, eval_df,
#     top_k_retrieve=10, top_n_rerank=5
# )

# print("\n--- Fine-tuned Bi-Encoder Only (FAISS metrics) ---")
# for metric in ['Recall@5 (FAISS)', 'Recall@10 (FAISS)', 'Recall@20 (FAISS)',
#                'MRR (FAISS)', 'NDCG@5 (FAISS)', 'NDCG@10 (FAISS)', 'NDCG@20 (FAISS)']:
#     print(f"  {metric:25s}: {ft_biencoder_summary[metric]:.4f}")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Evaluating FINE-TUNED bi-encoder ONLY (FAISS retrieval, no reranking)...



Evaluating: 100%|██████████| 500/500 [01:52<00:00,  4.44it/s]


--- Fine-tuned Bi-Encoder Only (FAISS metrics) ---
  Recall@5 (FAISS)         : 0.3280
  Recall@10 (FAISS)        : 0.4380
  Recall@20 (FAISS)        : 0.4380
  MRR (FAISS)              : 0.2230
  NDCG@5 (FAISS)           : 0.2379
  NDCG@10 (FAISS)          : 0.2736
  NDCG@20 (FAISS)          : 0.2736


In [ ]:
print("Evaluating PRE-TRAINED bi-encoder + PRE-TRAINED reranker...\n")
pt_summary, pt_details = evaluate_retrieval(
    pretrained_embed_model, pretrained_index, pretrained_reranker,
    all_chunks, chunk_metadata, eval_df
)

print("\n--- Pre-trained Results ---")
for metric, value in pt_summary.items():
    print(f"  {metric:25s}: {value:.4f}")

Evaluating PRE-TRAINED bi-encoder + PRE-TRAINED reranker...



Evaluating: 100%|█████████▉| 999/1000 [31:32<00:01,  1.90s/it]

In [ ]:
print("Evaluating FINE-TUNED bi-encoder + reranker...\n")
ft_summary, ft_details = evaluate_retrieval(
    finetuned_embed_model, finetuned_index, finetuned_reranker,
    all_chunks, chunk_metadata, eval_df
)

print("\n--- Fine-tuned Results ---")
for metric, value in ft_summary.items():
    print(f"  {metric:25s}: {value:.4f}")

### Step 5.3 — Comparison table

In [49]:
print("\n" + "=" * 75)
print("RETRIEVAL COMPARISON: Pre-trained vs. Fine-tuned")
print("=" * 75)
print(f"{'Metric':30s} | {'Pre-trained':>12s} | {'Fine-tuned':>12s} | {'Delta':>10s}")
print("-" * 75)

for metric in pt_summary:
    pt_val = pt_summary[metric]
    ft_val = ft_summary[metric]
    delta = ft_val - pt_val
    sign = '+' if delta >= 0 else ''
    print(f"  {metric:28s} | {pt_val:>11.4f}  | {ft_val:>11.4f}  | {sign}{delta:>9.4f}")

print("=" * 75)


RETRIEVAL COMPARISON: Pre-trained vs. Fine-tuned
Metric                         |  Pre-trained |   Fine-tuned |      Delta
---------------------------------------------------------------------------
  Recall@5 (FAISS)             |      0.3080  |      0.5340  | +   0.2260
  Recall@10 (FAISS)            |      0.4080  |      0.6620  | +   0.2540
  Recall@20 (FAISS)            |      0.5190  |      0.7410  | +   0.2220
  MRR (FAISS)                  |      0.2299  |      0.3739  | +   0.1440
  NDCG@5 (FAISS)               |      0.2298  |      0.3930  | +   0.1632
  NDCG@10 (FAISS)              |      0.2621  |      0.4345  | +   0.1724
  NDCG@20 (FAISS)              |      0.2901  |      0.4546  | +   0.1645
  Recall@5 (reranked)          |      0.3540  |      0.5080  | +   0.1540
  MRR (reranked)               |      0.2312  |      0.3623  | +   0.1311
  NDCG@5 (reranked)            |      0.2619  |      0.3987  | +   0.1368


### Step 5.4 — Breakdown by document type and question type

In [51]:
def print_breakdown(details_df, group_col, metric_col, label):
    """Print a breakdown of a metric by a grouping column."""
    if group_col not in details_df.columns:
        print(f"--- {label} --- (skipped, column '{group_col}' not available)")
        return
    print(f"--- {label} ---")
    grouped = details_df.groupby(group_col)[metric_col].agg(['mean', 'count'])
    grouped.columns = [metric_col, 'Count']
    grouped = grouped.sort_values('Count', ascending=False)
    print(grouped.to_string())

print("" + "=" * 60)
print("FINE-TUNED MODEL BREAKDOWN")
print("=" * 60)

print_breakdown(ft_details, 'doc_type', 'hit_at_5_reranked', 'Recall@5 (reranked) by Document Type')
print_breakdown(ft_details, 'question_type', 'hit_at_5_reranked', 'Recall@5 (reranked) by Question Type')
print_breakdown(ft_details, 'coverage_status', 'hit_at_5_reranked', 'Recall@5 (reranked) by Coverage Status')

FINE-TUNED MODEL BREAKDOWN
--- Recall@5 (reranked) by Document Type ---
                   hit_at_5_reranked  Count
doc_type                                   
CLAIMS_PROCESSING           0.364835    455
LCD                         0.614325    363
BENEFIT_POLICY              0.390000    100
NCD                         0.975610     82
--- Recall@5 (reranked) by Question Type ---
                     hit_at_5_reranked  Count
question_type                                
policy                        0.500000    286
ncd_reference                 0.582822    163
summary                       0.425676    148
use_case                      0.485149    101
service_condition             0.421053     95
criteria                      0.473118     93
yes_no                        0.593023     86
indications                   0.866667     15
mixed_policy                  0.700000     10
retired_status                1.000000      2
non_coverage_reason           1.000000      1
--- Recall@5 (reranke

In [53]:
print("Evaluating PRE-TRAINED bi-encoder + PRE-TRAINED reranker...\n")
pt_summary, pt_details = evaluate_retrieval(
    pretrained_embed_model, pretrained_index, pretrained_reranker,
    all_chunks, chunk_metadata, eval_df,
    top_k_retrieve=30, top_n_rerank=5
)

print("\n--- Pre-trained Results ---")
for metric, value in pt_summary.items():
    print(f"  {metric:25s}: {value:.4f}")

Evaluating PRE-TRAINED bi-encoder + PRE-TRAINED reranker...



Evaluating:   8%|▊         | 83/1000 [00:50<09:19,  1.64it/s]


KeyboardInterrupt: 

In [56]:
print("Evaluating FINE-TUNED bi-encoder + FINE-TUNED reranker...\n")
ft_summary, ft_details = evaluate_retrieval(
    finetuned_embed_model, finetuned_index, finetuned_reranker,
    all_chunks, chunk_metadata, eval_df,
    top_k_retrieve=10, top_n_rerank=5
)

print("\n--- Fine-tuned Results ---")
for metric, value in ft_summary.items():
    print(f"  {metric:25s}: {value:.4f}")

Evaluating FINE-TUNED bi-encoder + FINE-TUNED reranker...



Evaluating: 100%|██████████| 1000/1000 [03:46<00:00,  4.41it/s]


--- Fine-tuned Results ---
  Recall@5 (FAISS)         : 0.5340
  Recall@10 (FAISS)        : 0.6620
  Recall@20 (FAISS)        : 0.6620
  MRR (FAISS)              : 0.3637
  NDCG@5 (FAISS)           : 0.3930
  NDCG@10 (FAISS)          : 0.4345
  NDCG@20 (FAISS)          : 0.4345
  Recall@5 (reranked)      : 0.5340
  MRR (reranked)           : 0.3769
  NDCG@5 (reranked)        : 0.4159


In [57]:
print("Evaluating FINE-TUNED bi-encoder + FINE-TUNED reranker...\n")
ft_summary, ft_details = evaluate_retrieval(
    finetuned_embed_model, finetuned_index, finetuned_reranker,
    all_chunks, chunk_metadata, eval_df,
    top_k_retrieve=20, top_n_rerank=5
)

print("\n--- Fine-tuned Results ---")
for metric, value in ft_summary.items():
    print(f"  {metric:25s}: {value:.4f}")

Evaluating FINE-TUNED bi-encoder + FINE-TUNED reranker...



Evaluating: 100%|██████████| 1000/1000 [06:51<00:00,  2.43it/s]


--- Fine-tuned Results ---
  Recall@5 (FAISS)         : 0.5340
  Recall@10 (FAISS)        : 0.6620
  Recall@20 (FAISS)        : 0.7410
  MRR (FAISS)              : 0.3692
  NDCG@5 (FAISS)           : 0.3930
  NDCG@10 (FAISS)          : 0.4345
  NDCG@20 (FAISS)          : 0.4546
  Recall@5 (reranked)      : 0.5230
  MRR (reranked)           : 0.3700
  NDCG@5 (reranked)        : 0.4081


In [ ]:
print("Evaluating FINE-TUNED bi-encoder + FINE-TUNED reranker...\n")
ft_summary, ft_details = evaluate_retrieval(
    finetuned_embed_model, finetuned_index, finetuned_reranker,
    all_chunks, chunk_metadata, eval_df,
    top_k_retrieve=50, top_n_rerank=5
)

print("\n--- Fine-tuned Results ---")
for metric, value in ft_summary.items():
    print(f"  {metric:25s}: {value:.4f}")

In [ ]:
print("\n" + "=" * 75)
print("RETRIEVAL COMPARISON: Pre-trained vs. Fine-tuned")
print("=" * 75)
print(f"{'Metric':30s} | {'Pre-trained':>12s} | {'Fine-tuned':>12s} | {'Delta':>10s}")
print("-" * 75)

for metric in pt_summary:
    pt_val = pt_summary[metric]
    ft_val = ft_summary[metric]
    delta = ft_val - pt_val
    sign = '+' if delta >= 0 else ''
    print(f"  {metric:28s} | {pt_val:>11.4f}  | {ft_val:>11.4f}  | {sign}{delta:>9.4f}")

print("=" * 75)

In [ ]:
def print_breakdown(details_df, group_col, metric_col, label):
    """Print a breakdown of a metric by a grouping column."""
    if group_col not in details_df.columns:
        print(f"--- {label} --- (skipped, column '{group_col}' not available)")
        return
    print(f"--- {label} ---")
    grouped = details_df.groupby(group_col)[metric_col].agg(['mean', 'count'])
    grouped.columns = [metric_col, 'Count']
    grouped = grouped.sort_values('Count', ascending=False)
    print(grouped.to_string())

print("" + "=" * 60)
print("FINE-TUNED MODEL BREAKDOWN")
print("=" * 60)

print_breakdown(ft_details, 'doc_type', 'hit_at_5_reranked', 'Recall@5 (reranked) by Document Type')
print_breakdown(ft_details, 'question_type', 'hit_at_5_reranked', 'Recall@5 (reranked) by Question Type')
print_breakdown(ft_details, 'coverage_status', 'hit_at_5_reranked', 'Recall@5 (reranked) by Coverage Status')